In [1]:
library(dplyr)
library(Matrix)
library(data.table)
library(Seurat)
library(ggplot2)
library(RColorBrewer)
library(cowplot)
library(future)
library(viridis)
library(SeuratObject)
library(tidyr)
plan("multicore", workers = 16)
options(future.globals.maxSize = 100 * 1024^3)
library(ComplexHeatmap)
library(pheatmap)
library('stringr')

Warning message:
“package ‘dplyr’ was built under R version 4.2.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘Matrix’ was built under R version 4.2.3”

Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


The legacy packages maptools, rgdal, and rgeos, underpinning this package
will retire shortly. Please refer to R-spatial evolution reports on
https://r-spatial.org/r/2023/05/15/evolution4.html for details.
This package is now running under evolution status 0 

Attaching SeuratObject

Warning message:
“package ‘ggplot2’ was built under R version 4.2.3”
Warning message:
“package ‘viridis’ was built under R version 4.2.3”
Loading required package: viridisLite


Attaching package: ‘tidyr’


The following objects are masked from ‘package:Matrix’

In [3]:
seuobj = readRDS("/data/Bin200.merge.seuobj.sublayer.rds")

In [4]:
table(seuobj$sublayer)


L1_a L1_b   L2 L3_a L3_b   L4 L5_a L5_b L6_a L6_b   WM WM_a 
1024  816 2188 2074 2137 1223 1081 1557 2020 1566 3978 1226 

In [4]:
#### del sublyaer DEG
###################################
DefaultAssay(seuobj)="Spatial"

In [5]:
library(future)
options(future.globals.maxSize = 200 * 1024^3)
plan(multisession, workers=15)

In [7]:
Idents(seuobj)="Bayes.anno16"
seuobj = subset(seuobj,idents=c("L1","L2","L3","L4","L5","L6","WM"))

In [19]:
table(seuobj$Bayes.anno16)


  L1   L2   L3   L4   L5   L6   WM 
1840 2188 4211 1223 2638 3586 5204 

In [20]:
############# Calculate the layer DEG for each chip
##############################################
Idents(seuobj) = "T_names"
res.path = "/data/01.single.chip.layer.DEG/"
deg.all.raw = c()
deg.all = c()
for (T_name in unique(seuobj$T_names)){
    seuobj.sub = subset(seuobj,idents=T_name)
    seuobj.sub <- NormalizeData(object = seuobj.sub)
    Idents(seuobj.sub)="Bayes.anno16"
    scRNA1.markers <- FindAllMarkers(object = seuobj.sub,logfc.threshold = 0.05)
    scRNA1.markers$T_name = T_name
    write.csv(scRNA1.markers,paste0(res.path,"/",T_name,"_Bayes16.anno.Layer.DEG.raw.csv"))
    deg.all.raw = rbind(deg.all.raw,scRNA1.markers)
    ### filter
    scRNA1.markers = scRNA1.markers %>% subset(p_val_adj<0.05 & scRNA1.markers$avg_log2FC>0.2)
    write.csv(scRNA1.markers,paste0(res.path,"/",T_name,"_Bayes16.anno.Layer.DEG.logFC0.2.P0.05.csv"))
    ## merge.all
    deg.all = rbind(deg.all,scRNA1.markers)
    # #### TOP10
    # top10 <- scRNA1.markers %>% group_by(cluster) %>% top_n(n = 15, wt = avg_log2FC)
    # write.csv(top10,paste0(res.path,"/","Bayes",L,".subLayer.top15marker.csv"))
    }
write.csv(deg.all,paste0(res.path,"/","allchip_Bayes16.anno.Layer.DEG.logFC0.2.P0.05.csv"))
write.csv(deg.all.raw,paste0(res.path,"/","allchip_Bayes16.anno.Layer.DEG.P0.05.csv"))

Calculating cluster L1



In [32]:
###### Calculate the sublayer DEG for each chip
Idents(seuobj) = "T_names"
res.path = "/data/02.single.chip.Bayes.anno.sublayer.DEG"
deg.all = c()
deg.raw.all=c()
for (T_name in unique(seuobj$T_names)){
    seuobj.sub = subset(seuobj,idents=T_name)
    plan(multisession, workers=1)
    seuobj.sub <- NormalizeData(object = seuobj.sub)
    
    print(T_name)
    ### Calculate the deg of each sublayer of the chip
    Idents(seuobj.sub) = "Bayes.anno16"
    for (L in c("L1","L3","L5","L6","WM")){ 
        seuobj.sub.sub = subset(seuobj.sub,idents=L)
        seuobj.sub.sub <- NormalizeData(object = seuobj.sub.sub)
        Idents(seuobj.sub.sub)="sublayer"
        plan(multisession, workers=15)
        scRNA1.markers <- FindAllMarkers(object = seuobj.sub.sub,logfc.threshold = 0.05)
        scRNA1.markers$layer = L
        scRNA1.markers$T_name = T_name
        write.csv(scRNA1.markers,paste0(res.path,"/",T_name,"_Bayes16.anno.",L,".subLayer.DEG.raw.csv"))
        deg.raw.all = rbind(deg.raw.all,scRNA1.markers)
        ### filter
        scRNA1.markers = scRNA1.markers %>% subset(p_val_adj<0.05 & scRNA1.markers$avg_log2FC>0.2)
        write.csv(scRNA1.markers,paste0(res.path,"/",T_name,"_Bayes16.anno.",L,".subLayer.DEG.logFC0.2.P0.05.csv"))
        
        ####### Merge all sub-layer results.
        deg.all = rbind(deg.all,scRNA1.markers)
        #### TOP10
        # top10 <- scRNA1.markers %>% group_by(cluster) %>% top_n(n = 15, wt = avg_log2FC)
        # write.csv(top10,paste0(res.path,"/","Bayes",L,".subLayer.top15marker.csv"))
        }
    }
write.csv(deg.all,paste0(res.path,"/","allchip_Bayes16.anno.subLayer.DEG.logFC0.2.P0.05.csv"))
write.csv(deg.raw.all,paste0(res.path,"/","allchip_Bayes16.anno.subLayer.DEG.P0.05.csv"))

[1] "T756"


Calculating cluster WM_a

Calculating cluster WM



[1] "T928"


Calculating cluster WM_a

Calculating cluster WM



[1] "T989"


Calculating cluster WM_a

Calculating cluster WM



[1] "T993"


Calculating cluster WM_a

Calculating cluster WM



[1] "T996"


Calculating cluster WM_a

Calculating cluster WM



In [37]:
###############  merge all chip sublayer raw deg
sublayer.deg.path="/data/02.single.chip.Bayes.anno.sublayer.DEG/"
file.list = list.files(sublayer.deg.path,pattern="subLayer.DEG.raw.csv")
deg.all =c()
for(i in file.list){
    deg.i = read.csv(paste0(sublayer.deg.path,"/",i),row.names=1)
    deg.all = rbind(deg.all,deg.i)
    }
deg.all = deg.all %>% subset(p_val_adj<0.05)
write.csv(deg.all,"/data/allchip_Bayes16.anno.subLayer.DEG.P0.05.csv")
deg.all = deg.all %>% subset(p_val_adj<0.05 & avg_log2FC>0.2)
write.csv(deg.all,"/data/allchip_Bayes16.anno.subLayer.DEG.logFC0.2.P0.05.csv")

In [39]:
head(deg.all)
table(deg.all$layer)
table(deg.all$cluster)

,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,gene,layer,T_name
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>
GFAP,3.969219e-21,2.009335,0.972,0.991,1.942337e-16,L1_a,GFAP,L1,T756
LINC01411,9.056817e-14,1.680574,0.505,0.057,4.431953e-09,L1_a,LINC01411,L1,T756
CRYAB,7.221271e-12,1.825792,0.872,0.858,3.533729e-07,L1_a,CRYAB,L1,T756
AQP4,7.963042e-12,1.404281,0.908,0.896,3.896715e-07,L1_a,AQP4,L1,T756
S100B,5.082961e-10,1.052535,0.899,0.962,2.487347e-05,L1_a,S100B,L1,T756
DBI,1.963528e-09,1.207031,0.872,0.943,9.608525e-05,L1_a,DBI,L1,T756



  L1   L3   L5   L6   WM 
2929  761  484 1832 5599 


L1_a L1_b L3_a L3_b L5_a L5_b L6_a L6_b   WM WM_a 
 775 2154  413  348  282  202 1135  697 2445 3154 

In [ ]:
##############################  For each chip, the intersection of the layer's DEG and the corresponding sublayer's DEG is determined and designated as that sublayer's DEG for that specific chip.
#
####################################################################################################################

In [67]:
layer.deg.raw.all = read.csv("/data/allchip_Bayes16.anno.Layer.DEG.P0.05.csv",row.names=1)
sublayer.raw.deg.all = read.csv("/data/allchip_Bayes16.anno.subLayer.DEG.P0.05.csv",row.names=1)

In [68]:
min(abs(layer.deg.all$avg_log2FC))
min(abs(sublayer.deg.all$avg_log2FC))

[1] 0.1000079

[1] 0.1000209

In [25]:
for (logFC.filter in c("0.1","0.15","0.2","0.25")){
    ############ Filter by logFC
    layer.deg.all = layer.deg.raw.all %>% filter(p_val_adj<0.05) %>% filter(avg_log2FC>logFC.filter)
    sublayer.deg.all = sublayer.raw.deg.all %>% filter(p_val_adj<0.05) %>% filter(avg_log2FC>logFC.filter)
    
    ###### Take the intersection of the DEGs at each layer and the DEGs at each sub-layer.
    ###############################################
    sublayer.marker.all = c()
    for (T_name.i in unique(layer.deg.all$T_name)){
        layer.deg.all.T_name = layer.deg.all %>% filter(T_name %in% T_name.i)
        sublayer.deg.all.T_name = sublayer.deg.all %>% filter(T_name %in% T_name.i)
        ### Filter by layer
        for (L in c("L1","L3","L5","L6")){
            layer.deg.all.T_name.L = layer.deg.all.T_name %>% filter(T_name %in% T_name.i)%>% filter(cluster %in% L)
            sublayer.deg.all.T_name.L.inlayer = sublayer.deg.all.T_name %>% filter(T_name %in% T_name.i)%>% filter(layer %in% L) %>% filter(gene %in% layer.deg.all.T_name.L$gene)
            print(dim(sublayer.deg.all.T_name.L.inlayer))
            sublayer.marker.all = rbind(sublayer.marker.all,sublayer.deg.all.T_name.L.inlayer)
            }
        }
    write.csv(sublayer.marker.all,paste0("Layer.and.sublayer.marker.intersect.logFC",logFC.filter,".csv"))
    ######### Filter for genes appearing on at least one chip.
    ###############################################
    conserved.gene =c()
    sublayer.marker.all$layer_cluster = paste0(sublayer.marker.all$layer,"_",sublayer.marker.all$cluster)
    for (i in unique(sublayer.marker.all$layer_cluster)){
        sublayer.marker.i = sublayer.marker.all %>% filter(layer_cluster %in% i)
        gene.conserved = as.data.frame(table(sublayer.marker.i$gene)) %>% filter(Freq >1)
        if (dim(gene.conserved)[1]>0){
            gene.conserved$layer_cluster = i
            conserved.gene = rbind(conserved.gene,gene.conserved)
            }
        }
    ### Extract the layer and cluster information.
    conserved.gene$Layer <- sapply(str_split(conserved.gene$layer_cluster, "_"), function(x) head(x, 1))  
    conserved.gene$cluster<-sapply(str_split(conserved.gene$layer_cluster, "_"), function(x) tail(x, 1))  
    write.csv(conserved.gene,paste0("sublayer.marker.logFC",logFC.filter,".csv"))
    }


[1] 50  9
[1] 36  9
[1] 20  9
[1] 100   9
[1] 33  9
[1] 121   9
[1] 1 9
[1] 14  9
[1] 111   9
[1] 230   9
[1] 85  9
[1] 119   9
[1] 555   9
[1] 195   9
[1] 188   9
[1] 417   9
[1] 233   9
[1] 240   9
[1] 161   9
[1] 126   9
[1] 47  9
[1] 21  9
[1] 16  9
[1] 58  9
[1] 32  9
[1] 96  9
[1] 0 9
[1] 7 9
[1] 104   9
[1] 103   9
[1] 39  9
[1] 39  9
[1] 438   9
[1] 110   9
[1] 110   9
[1] 163   9
[1] 183   9
[1] 93  9
[1] 77  9
[1] 54  9
[1] 40  9
[1] 11  9
[1] 11  9
[1] 23  9
[1] 32  9
[1] 56  9
[1] 0 9
[1] 3 9
[1] 82  9
[1] 35  9
[1] 15  9
[1] 13  9
[1] 324   9
[1] 54  9
[1] 42  9
[1] 51  9
[1] 135   9
[1] 27  9
[1] 34  9
[1] 25  9
[1] 33  9
[1] 5 9
[1] 6 9
[1] 11  9
[1] 25  9
[1] 34  9
[1] 0 9
[1] 1 9
[1] 58  9
[1] 10  9
[1] 2 9
[1] 4 9
[1] 234   9
[1] 22  9
[1] 16  9
[1] 19  9
[1] 93  9
[1] 12  9
[1] 13  9
[1] 14  9


In [ ]:
######### Sublayer Gene Module Score Analysis

In [29]:
cols =c("#d473d4","#cd1076",
        "#377eb9",
        "#006400","#4dae48",
        "#974f9f",
        "#cdcd00","#ffff00",
        "#ff7f24","#ffa700",
        "#59260b","#a65628")
names(cols) =c("L1_b","L1_a",
        "L2",
        "L3_a","L3_b",
        "L4",
        "L5_a","L5_b",
        "L6_a","L6_b",
        "WM_a","WM")

In [26]:
logFC.filter = "0.1"
    conserved.gene=read.csv(paste0("/data/filter.LOGfc01.Freqbig1.gene.csv"),row.names=1)
    colnames(conserved.gene)[1]="gene"
    pdf(paste0("/data/sublayer.marker.logFC",logFC.filter,"Freqbig1.module.score.pdf"),height=5)
    ############ Iteratively generate violin plots of module scores for DEGs within each sublayer.
    for (i in  unique(conserved.gene$layer_cluster)){
        gene.i = conserved.gene%>% filter(layer_cluster %in% i)
        gene.i = list(gene.i$gene)
        seuobj.score=AddModuleScore(seuobj,features = gene.i,name = i)
        g = paste0(i,"1")
        data = seuobj.score@meta.data[,c("sublayer","Bayes.anno16",g)]
        colnames(data) =c("cluster","layer","score")

        p = ggplot(data,aes(x=factor(cluster),y=score,fill=cluster))+
          geom_violin(scale = "width",alpha=0.8,width=0.5,size=0.8)+ 
          scale_fill_manual(values = cols)+   xlab("")+  ylab("")+    theme_bw()+         
          theme(panel.grid.major=element_blank(),     panel.grid.minor=element_blank(), panel.border=element_rect(size=0.2),                 
            axis.text.x = element_text(angle=60,size=10,vjust = 1,hjust =1,color = "black"),  axis.text.y = element_text(size =10),
            legend.position = c(1.1,0.25) )+ggtitle(i)
        print(p)
        }
    dev.off()

Warning message:
“Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
ℹ Please use `linewidth` instead.”
Warning message:
“The `size` argument of `element_rect()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.”


png 
  2

In [51]:
head(seuobj.score)

,orig.ident,nCount_Spatial,nFeature_Spatial,area,percent.mt,nCount_SCT,nFeature_SCT,SCT_snn_res.0.5,SCT_snn_res.0.8,SCT_snn_res.1.2,⋯,Bayes15,Bayes17,Bayes18,Bayes19,Bayes21,Bayes22,Bayes23,Bayes.anno16,sublayer,L5_a1
,<chr>,<dbl>,<int>,<chr>,<dbl>,<dbl>,<int>,<fct>,<chr>,<chr>,⋯,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<chr>,<chr>,<dbl>
T756_BIN200_27,BIN200,591,309,L6,18.78173,23535,7266,7,4,8,⋯,14,14,15,16,17,21,22,L1,L1_a,0.02579598
T756_BIN200_28,BIN200,1827,809,L6,24.52107,24052,6444,5,4,8,⋯,1,1,1,4,1,1,1,WM,WM_a,0.04423533
T756_BIN200_29,BIN200,3296,1074,WM,39.13835,23920,5625,5,4,8,⋯,1,1,1,4,1,1,1,WM,WM_a,0.17473484
T756_BIN200_30,BIN200,8327,2330,WM,36.43569,24431,4172,5,4,8,⋯,1,1,1,4,1,1,1,WM,WM_a,0.14631183
T756_BIN200_31,BIN200,16086,4346,WM,25.73667,25626,4496,6,4,8,⋯,8,8,1,1,1,1,1,L6,L6_b,0.27093192
T756_BIN200_32,BIN200,16379,4360,WM,26.56450,25540,4462,5,4,8,⋯,8,8,1,1,1,1,1,L6,L6_b,0.32330665
T756_BIN200_33,BIN200,16897,4126,WM,30.45511,25393,4190,5,4,8,⋯,8,8,1,1,9,9,9,L6,L6_b,0.38205781
T756_BIN200_34,BIN200,13358,3562,WM,31.35200,25139,3984,3,4,8,⋯,8,8,8,1,9,9,9,L6,L6_b,0.41495720
T756_BIN200_35,BIN200,12345,3461,WM,30.87080,24943,4070,5,4,8,⋯,8,8,8,1,9,9,9,L6,L6_b,0.34557751


In [27]:
logFC.filter = "0.1"
    conserved.gene=read.csv(paste0("/data/filter.LOGfc01.Freqbig1.gene.csv"),row.names=1)
    colnames(conserved.gene)[1]="gene"
    pdf(paste0("/data/sublayer.marker.logFC",logFC.filter,"Freqbig1.module.score.STEREumap.pdf"),height=5)
    
############ Iteratively generate spatial expression plots of module scores for DEGs within each sublayer.
    for (i in  unique(conserved.gene$layer_cluster)){
        gene.i = conserved.gene%>% filter(layer_cluster %in% i)
        gene.i = list(gene.i$gene)
        seuobj.score=AddModuleScore(seuobj,features = gene.i,name = i)
        g = paste0(i,"1")
          
        allmean = data.frame(seuobj.score@meta.data[,c("coor_x","coor_y",g)])
        colnames(allmean) <- c("coor_x","coor_y","Exp")
        options(repr.plot.width=10, repr.plot.height=6) 
        p = ggplot()+geom_point(data=allmean,aes(x=coor_x,y=coor_y,color=Exp),size=0.3, alpha=1.3,stroke = 0)+ scale_colour_gradientn(values = seq(0,1,0.2),colours = c('black','blue','green','yellow','orange','red'))+
                theme_classic()+coord_fixed()+theme(panel.background = element_rect(fill="white"))+#labs(title = paste0(j,"_",gene))+ggtitle(paste0(j,"_",i))+
                theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank(),panel.border = element_blank(),axis.line = element_blank(),
                    axis.text.x = element_blank(), axis.text.y = element_blank(),axis.title.x = element_blank(),axis.title.y = element_blank(),axis.ticks.x =element_blank(),axis.ticks.y =element_blank())+ggtitle(paste0(i)) ##去掉xy坐标
            print(p)
        }
    dev.off()
 #   }  

png 
  2